In [1]:
import pandas as pd
import numpy as np

In [23]:
from time import sleep, time

In [3]:
from epics import PV

In [4]:
PCs = [PV('BME%dVV:setVolt' % (i+1)) for i in range(3)]

In [36]:
cur = PV('CUM1ZK3RP:rdCur')

In [5]:
for PC in PCs:
    print('---------------')
    print(PC.info)
    print(PC.get())    
    print(type(PC.get()))

---------------
== BME1VV:setVolt  (time_double) ==
   value      = 0
   char_value = '0'
   count      = 1
   nelm       = 1
   type       = time_double
   units      = V
   precision  = 0
   host       = iocsc3cp.mlscs.bessy.de:34125
   access     = read/write
   status     = 0
   severity   = 0
   timestamp  = 1693144602.017 (2023-08-27 15:56:42.01699)
   posixseconds        = 1693144602.0
   nanoseconds= 16993069
   upper_ctrl_limit    = 9000.0
   lower_ctrl_limit    = 0.0
   upper_disp_limit    = 9000.0
   lower_disp_limit    = 0.0
   upper_alarm_limit   = nan
   lower_alarm_limit   = nan
   upper_warning_limit = nan
   lower_warning_limit = nan
   PV is internally monitored, with 0 user-defined callbacks:
0.0
<class 'float'>
---------------
== BME2VV:setVolt  (time_double) ==
   value      = 0
   char_value = '0'
   count      = 1
   nelm       = 1
   type       = time_double
   units      = V
   precision  = 0
   host       = iocsc3cp.mlscs.bessy.de:34125
   access     = read/wr

In [37]:
cur.get()

130.29086241899844

## simple scan...

In [9]:
voltlist = np.arange(0,9001,100)
print(len(voltlist))
voltlist

91


array([   0,  100,  200,  300,  400,  500,  600,  700,  800,  900, 1000,
       1100, 1200, 1300, 1400, 1500, 1600, 1700, 1800, 1900, 2000, 2100,
       2200, 2300, 2400, 2500, 2600, 2700, 2800, 2900, 3000, 3100, 3200,
       3300, 3400, 3500, 3600, 3700, 3800, 3900, 4000, 4100, 4200, 4300,
       4400, 4500, 4600, 4700, 4800, 4900, 5000, 5100, 5200, 5300, 5400,
       5500, 5600, 5700, 5800, 5900, 6000, 6100, 6200, 6300, 6400, 6500,
       6600, 6700, 6800, 6900, 7000, 7100, 7200, 7300, 7400, 7500, 7600,
       7700, 7800, 7900, 8000, 8100, 8200, 8300, 8400, 8500, 8600, 8700,
       8800, 8900, 9000])

In [40]:
timelist = []
curlist  = []
BMElist  = []
BMEnlist = []
for bme in range(3):
    for PC, v0 in zip(PCs, [8350,8800,8100]):
        PC.put(v0)
    sleep(3)
    for i,v in enumerate(voltlist):
        sleep(0.5)
        PCs[bme].put(v)
        sleep(14.5)
        timelist.append(time())
        curlist.append(cur.get())
        BMElist.append(v)
        BMEnlist.append(bme+1)
for PC, v0 in zip(PCs, [8350,8800,8100]):
    PC.put(v0)

In [42]:
caldf = pd.DataFrame({'timestamp':timelist, 'current':curlist, 'PC_number':BMEnlist, 'PC_voltage':BMElist})
caldf

,PC_number,PC_voltage,current,timestamp
0,1,0,125.519015,1.694686e+09
1,1,100,125.443831,1.694686e+09
2,1,200,125.372056,1.694686e+09
3,1,300,125.299988,1.694686e+09
4,1,400,125.221325,1.694686e+09
5,1,500,125.145096,1.694686e+09
6,1,600,125.065336,1.694686e+09
7,1,700,124.998187,1.694686e+09
8,1,800,124.927439,1.694686e+09
9,1,900,124.848483,1.694686e+09


In [46]:
caldf.to_hdf('PCcalib_20230914.hdf5', 'data')

In [22]:
time()/3600/24/365.25

53.70107137769208

In [ ]:
scope +2:49 min

## with live evaluation...

In [6]:
detectornames = ['SCOPE1ZULP:h%dp%d:rdAmplAv' % (h+1, p+1) for h in range(2) for p in range(3)]
print(detectornames)

['SCOPE1ZULP:h1p1:rdAmplAv', 'SCOPE1ZULP:h1p2:rdAmplAv', 'SCOPE1ZULP:h1p3:rdAmplAv', 'SCOPE1ZULP:h2p1:rdAmplAv', 'SCOPE1ZULP:h2p2:rdAmplAv', 'SCOPE1ZULP:h2p3:rdAmplAv']


In [7]:
detectoravgs    = [PV(name) for name in detectornames]
detectormonitor = PV('SCOPE1ZULP:h1p1:rdAmpl')
detectoravglen  = PV('SCOPE1ZULP:rdAvLength')

In [8]:
print(detectormonitor.get())
print(detectoravglen.get())

0.00390625
50.0


In [9]:
for d in detectoravgs:
    print(d.get())

0.0018132812500000004
0.00122109375
0.0008437500000000001
0.3226953125
0.322890625
0.3241015625


In [10]:
ringcurrent = PV('CUM1ZK3RP:rdCur')
print(ringcurrent.get())

124.96442388585332


In [15]:
class Pockelscalib:
    def __init__(self, PC_PVs, det_PVs, det_names, det_avglen, det_monitor, voltagestandard = [8350, 8800, 8100]):
        self.__PCs = PC_PVs
        self.__dets = det_PVs
        self.__detnames = det_names
        self.__det_avglen = det_avglen
        self.__det_monitor = det_monitor
        self.voltagestandard = voltagestandard
        self.__ringcurrent = PV('CUM1ZK3RP:rdCur')
        
    def record_calibration(self, outputfilename=None, avglen_target=50, voltagestepsize=250):
        self.voltageset = np.arange(0,9000+voltagestepsize, voltagestepsize)
        self.averagelen_target = avglen_target
        filename = outputfilename
        self.next = False
        self.count = 0
        self.averagelen_target = 50
        self.current_PC = 0
        self.current_volt = 0
        self.finished = False
        self.result_df = pd.DataFrame()
        for PC, standard in zip(self.__PCs, self.voltagestandard):
            PC.put(standard) # all PCs to standard voltages
        sleep(0.1)
        self.__PCs[0].put(self.voltageset[0]) # put first PC to first voltage step
        sleep(1)
        self.__det_monitor.add_callback(self._callback) #register callback and start taking data!

        while not self.finished: #wait until we are finished
            sleep(0.1)
            if self.next:
                print(self.__PCs[self.current_PC].put(self.voltageset[self.current_volt]))
                self.next = False

        self.__det_monitor.clear_callbacks() # delete callback
        self.result_df.to_hdf('/net/nfs/srv/OPI/MachinePhysics/MachineDevelopment/kruschinski/ssmb/LiveEvaluation/Pockels_cell_calibration/' + filename, key='data')
        return self.result_df
        
    def _callback(self, **kwargs):
        if not self.finished:
            self.count += 1
        if self.count >= self.averagelen_target and self.__det_avglen.get() >= self.averagelen_target:
            # we have collected enough data, save the current average and continue to the next point.
            outd = {'PC%d_setVolt' % (i+1): PC.get() for i, PC in enumerate(self.__PCs)}
            outd.update({detname: detavg.get() for detname, detavg in zip(self.__detnames, self.__dets)})
            outd.update({'avglen':  self.__det_avglen.get(), 'ringcurrent': self.__ringcurrent.get()})
            print(outd)
            self.result_df = pd.concat([self.result_df, pd.DataFrame(outd, index=[0])], ignore_index = True)

            self.count = 0
            self.current_volt += 1
            if self.current_volt >= len(self.voltageset):
                # we are done with the voltage scan, go to the next PC
                self.current_volt = 0
                self.current_PC += 1
                for PC, standard in zip(self.__PCs, self.voltagestandard):
                    PC.put(standard) # all PCs to standard voltages
                sleep(0.1)
                if self.current_PC >= len(self.__PCs):
                    # we are done with all PCs and are finished!
                    self.finished = True
                    return
            print(self.current_PC)
            print(self.voltageset[self.current_volt])
            self.next=True # set current PC to the next voltage setpoint

In [16]:
PC_calibration = Pockelscalib(PCs, detectoravgs, detectornames, detectoravglen, detectormonitor)

In [17]:
#calibdf = PC_calibration.record_calibration('20230310_Pockels_calibration.hdf5')
#print(calibdf)

{'PC2_setVolt': 8800.0, 'SCOPE1ZULP:h1p3:rdAmplAv': 0.00092578125, 'PC3_setVolt': 8100.0, 'SCOPE1ZULP:h1p2:rdAmplAv': 0.0010382812500000001, 'PC1_setVolt': 0.0, 'avglen': 50.0, 'ringcurrent': 124.74529240073542, 'SCOPE1ZULP:h2p2:rdAmplAv': 0.326875, 'SCOPE1ZULP:h1p1:rdAmplAv': 0.0007171874999999999, 'SCOPE1ZULP:h2p1:rdAmplAv': 0.32828125, 'SCOPE1ZULP:h2p3:rdAmplAv': 0.326640625}
0
250
1
{'PC2_setVolt': 8800.0, 'SCOPE1ZULP:h1p3:rdAmplAv': 0.0008054687500000001, 'PC3_setVolt': 8100.0, 'SCOPE1ZULP:h1p2:rdAmplAv': 0.00066015625, 'PC1_setVolt': 250.0, 'avglen': 50.0, 'ringcurrent': 124.62867019581722, 'SCOPE1ZULP:h2p2:rdAmplAv': 0.32515625, 'SCOPE1ZULP:h1p1:rdAmplAv': 0.0006929687500000002, 'SCOPE1ZULP:h2p1:rdAmplAv': 0.3275390625, 'SCOPE1ZULP:h2p3:rdAmplAv': 0.3265625}
0
500
1
{'PC2_setVolt': 8800.0, 'SCOPE1ZULP:h1p3:rdAmplAv': 0.0010749999999999998, 'PC3_setVolt': 8100.0, 'SCOPE1ZULP:h1p2:rdAmplAv': 0.0010765624999999996, 'PC1_setVolt': 500.0, 'avglen': 50.0, 'ringcurrent': 124.5067280510

NameError: name 'current_PC' is not defined

{'PC2_setVolt': 8800.0, 'SCOPE1ZULP:h1p3:rdAmplAv': 0.04196874999999999, 'PC3_setVolt': 8100.0, 'SCOPE1ZULP:h1p2:rdAmplAv': 0.040281250000000005, 'PC1_setVolt': 9000.0, 'avglen': 50.0, 'ringcurrent': 120.69412658908585, 'SCOPE1ZULP:h2p2:rdAmplAv': 0.314375, 'SCOPE1ZULP:h1p1:rdAmplAv': 0.041195312500000004, 'SCOPE1ZULP:h2p1:rdAmplAv': 0.3173828125, 'SCOPE1ZULP:h2p3:rdAmplAv': 0.3138671875}
1
250
1
{'PC2_setVolt': 250.0, 'SCOPE1ZULP:h1p3:rdAmplAv': 0.043140625000000016, 'PC3_setVolt': 8100.0, 'SCOPE1ZULP:h1p2:rdAmplAv': 0.04315625000000001, 'PC1_setVolt': 8350.0, 'avglen': 50.0, 'ringcurrent': 120.58799857250631, 'SCOPE1ZULP:h2p2:rdAmplAv': 0.32125, 'SCOPE1ZULP:h1p1:rdAmplAv': 0.04409375, 'SCOPE1ZULP:h2p1:rdAmplAv': 0.319375, 'SCOPE1ZULP:h2p3:rdAmplAv': 0.318359375}
1
500
1
{'PC2_setVolt': 500.0, 'SCOPE1ZULP:h1p3:rdAmplAv': 0.0019390625, 'PC3_setVolt': 8100.0, 'SCOPE1ZULP:h1p2:rdAmplAv': 0.0019210937499999996, 'PC1_setVolt': 8350.0, 'avglen': 50.0, 'ringcurrent': 120.48027089490809, 'SCO

NameError: name 'current_PC' is not defined

{'PC2_setVolt': 9000.0, 'SCOPE1ZULP:h1p3:rdAmplAv': 0.042359375000000005, 'PC3_setVolt': 8100.0, 'SCOPE1ZULP:h1p2:rdAmplAv': 0.04070312499999999, 'PC1_setVolt': 8350.0, 'avglen': 50.0, 'ringcurrent': 116.92595802970042, 'SCOPE1ZULP:h2p2:rdAmplAv': 0.3071484375, 'SCOPE1ZULP:h1p1:rdAmplAv': 0.0425703125, 'SCOPE1ZULP:h2p1:rdAmplAv': 0.305703125, 'SCOPE1ZULP:h2p3:rdAmplAv': 0.3058203125}
2
250
1
{'PC2_setVolt': 8800.0, 'SCOPE1ZULP:h1p3:rdAmplAv': 0.04221093750000002, 'PC3_setVolt': 250.0, 'SCOPE1ZULP:h1p2:rdAmplAv': 0.04334375000000001, 'PC1_setVolt': 8350.0, 'avglen': 50.0, 'ringcurrent': 116.82421105346361, 'SCOPE1ZULP:h2p2:rdAmplAv': 0.3045703125, 'SCOPE1ZULP:h1p1:rdAmplAv': 0.042296875000000005, 'SCOPE1ZULP:h2p1:rdAmplAv': 0.306171875, 'SCOPE1ZULP:h2p3:rdAmplAv': 0.3022265625}
2
500
1
{'PC2_setVolt': 8800.0, 'SCOPE1ZULP:h1p3:rdAmplAv': 0.001396875, 'PC3_setVolt': 500.0, 'SCOPE1ZULP:h1p2:rdAmplAv': 0.0014406249999999998, 'PC1_setVolt': 8350.0, 'avglen': 50.0, 'ringcurrent': 116.72103806

KeyboardInterrupt: 

In [18]:
detectormonitor.clear_callbacks() # delete callback


In [16]:
#PCs[0].put(8350.0)

1

In [21]:
PC_calibration.result_df.to_hdf('/net/nfs/srv/OPI/MachinePhysics/MachineDevelopment/kruschinski/ssmb/LiveEvaluation/Pockels_cell_calibration/20230310_Pockels_calibration.hdf5', key='data')